## Instalación de dependencias (en dos pasos para evitar conflictos con `requests`)

In [ ]:
# 1. Instalar dependencias
!pip -q install -U fastapi uvicorn pyngrok langchain langchain-openrouter langchain-classic

In [ ]:
!pip -q install requests==2.32.4 --upgrade --force-reinstall --no-deps

**Nota:** Si al instalar requests aparece un error, vuelve a ejecutar la celda.

## Imports y configuración de secretos

In [ ]:
# 2. Importar librerías y configurar
import os
import json
import requests
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from google.colab import userdata
from langchain_openrouter import ChatOpenRouter
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [ ]:
# Configurar OpenRouter
openrouter_api_key = userdata.get("OPENROUTER_API_KEY")
if not openrouter_api_key:
    raise ValueError("Agrega OPENROUTER_API_KEY en los Secrets de Colab.")
os.environ["OPENROUTER_API_KEY"] = openrouter_api_key

In [ ]:
# Crear el modelo de generación (temperatura baja para respuestas deterministas)
llm = ChatOpenRouter(
    model="google/gemini-2.5-flash-lite",
    temperature=0.2,
)

In [ ]:
# 3. Crear la aplicación FastAPI
app = FastAPI(
    title="Agente con herramientas (clima y chistes)",
    description="API con un agente que decide qué herramienta usar según la pregunta del usuario.",
    version="1.0.0",
)

print("✅ Entorno listo. Ahora definiremos las herramientas y el agente.")

## Definir los modelos Pydantic

In [ ]:
# 4. Modelos Pydantic
class AgentRequest(BaseModel):
    message: str = Field(..., min_length=2, description="Mensaje del usuario en español")

class AgentResponse(BaseModel):
    message: str
    tool_used: str
    raw_response: str

## Implementar la herramienta `get_weather`

In [ ]:
# 5. Herramienta: get_weather
@tool
def get_weather(city: str) -> str:
    """
    Obtiene el clima actual de una ciudad usando Open‑Meteo.
    Args:
        city: Nombre de la ciudad (ej. "Madrid", "Buenos Aires")
    Returns:
        Mensaje con el clima en español.
    """
    try:
        # 1. Geocodificar la ciudad
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_response = requests.get(geo_url)
        geo_data = geo_response.json()

        if not geo_data.get("results"):
            return f"No se encontró la ciudad '{city}'. Por favor, verifica el nombre."

        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        city_name = geo_data["results"][0]["name"]
        country = geo_data["results"][0].get("country", "")

        # 2. Obtener el clima actual
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        weather_response = requests.get(weather_url)
        weather_data = weather_response.json()

        if "current_weather" not in weather_data:
            return f"No se pudo obtener el clima para {city_name}."

        current = weather_data["current_weather"]
        temp = current["temperature"]
        weather_code = current["weathercode"]

        # Mapear código de clima a descripción (simplificado)
        weather_map = {
            0: "cielo despejado",
            1: "mayormente despejado",
            2: "parcialmente nublado",
            3: "nublado",
            45: "niebla",
            48: "niebla con escarcha",
            51: "llovizna ligera",
            53: "llovizna moderada",
            55: "llovizna intensa",
            61: "lluvia ligera",
            63: "lluvia moderada",
            65: "lluvia intensa",
            71: "nieve ligera",
            73: "nieve moderada",
            75: "nieve intensa",
            80: "chubascos ligeros",
            81: "chubascos moderados",
            82: "chubascos intensos",
            95: "tormenta eléctrica",
            96: "tormenta con granizo ligero",
            99: "tormenta con granizo intenso"
        }
        description = weather_map.get(weather_code, "condiciones climáticas variables")

        return f"En {city_name} ({country}) hay {temp}°C y {description}."

    except Exception as e:
        return f"Error al obtener el clima: {str(e)}"

## Implementar la herramienta `tell_joke`

In [ ]:
# 6. Herramienta: tell_joke
@tool
def tell_joke() -> str:
    """
    Cuenta un chiste gracioso. Si la API falla, usa un chiste local.
    Returns:
        Un chiste en español.
    """
    try:
        # Intentar obtener chiste de la API
        joke_url = "https://v2.jokeapi.dev/joke/Any?lang=es&safe-mode=true"
        response = requests.get(joke_url, timeout=5)

        if response.status_code == 200:
            data = response.json()
            if data.get("type") == "single":
                return data["joke"]
            elif data.get("type") == "twopart":
                return f"{data['setup']} - {data['delivery']}"

        # Fallback local si la API falla
        fallback_jokes = [
            "¿Qué le dice una taza a otra taza? ¡Nos vemos en la mesa!",
            "¿Cómo se llama el campeón de buceo japonés? Tokofondo.",
            "¿Qué hace una abeja en el gimnasio? ¡Zum-ba!",
            "¿Cuál es el colmo de un eléctrico? Que su mujer sea la corriente.",
            "¿Qué le dice un árbol a otro árbol? ¡Qué tronco eres!"
        ]
        import random
        return random.choice(fallback_jokes)

    except Exception:
        # Fallback local si hay cualquier error
        fallback_jokes = [
            "¿Qué le dice una taza a otra taza? ¡Nos vemos en la mesa!",
            "¿Cómo se llama el campeón de buceo japonés? Tokofondo."
        ]
        import random
        return random.choice(fallback_jokes)

## Crear el agente con LangChain

In [ ]:
# 7. Registrar las herramientas
tools = [get_weather, tell_joke]

In [ ]:
# 8. Configurar el prompt del agente
prompt = ChatPromptTemplate.from_messages([
    ("system", """
Eres un asistente útil que responde preguntas en español.

Tienes acceso a las siguientes herramientas:
- get_weather(city): Úsala cuando el usuario pregunte por el clima de una ciudad.
- tell_joke(): Úsala cuando el usuario pida un chiste.

Siempre responde en español. Sé amable y conciso.
Explica brevemente qué herramienta usaste y por qué.
"""),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [ ]:
# 9. Crear el agente y el ejecutor
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,  # Muestra los pasos del agente en el notebook (educativo)
    handle_parsing_errors=True,
    max_iterations=5,
)
print("✅ Agente creado correctamente.")

## Implementar el endpoint `POST /agent`

In [ ]:
# 10. Endpoint POST /agent
@app.post("/agent", response_model=AgentResponse)
def agent_endpoint(request: AgentRequest):
    """
    Envía un mensaje al agente. El agente decide qué herramienta usar.
    """
    try:
        # Invocar al agente
        result = agent_executor.invoke({"input": request.message})

        # Extraer la respuesta y la herramienta utilizada
        output = result.get("output", "No se pudo obtener una respuesta.")

        # Intentar identificar qué herramienta se usó (para el campo tool_used)
        # Buscar en el historial del agente (si está disponible)
        tool_used = "unknown"
        if "intermediate_steps" in result:
            for step in result["intermediate_steps"]:
                if len(step) >= 2:
                    action = step[0]
                    if hasattr(action, "tool"):
                        tool_used = action.tool
                        break

        # Si no se pudo identificar, intentar inferir por el contenido
        if tool_used == "unknown" or tool_used == "_tools":
            if "get_weather" in output.lower():
                tool_used = "get_weather"
            elif "chiste" in output.lower():
                tool_used = "tell_joke"

        return AgentResponse(
            message=output,
            tool_used=tool_used,
            raw_response=output
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error al procesar la solicitud: {str(e)}")

## Levantar la API y exponerla con ngrok

In [ ]:
# 11. Levantar servidor
import threading
import time
import uvicorn

PORT = 8000

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(2)
print(f"✅ Servidor iniciado en el puerto {PORT}")

In [ ]:
# 12. Abrir túnel con ngrok
from google.colab import userdata
from pyngrok import ngrok

ngrok_authtoken = userdata.get("NGROK_AUTHTOKEN")
if not ngrok_authtoken:
    raise ValueError("Agrega NGROK_AUTHTOKEN en los Secrets de Colab.")

ngrok.set_auth_token(ngrok_authtoken)
ngrok.kill()
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print("🌍 URL pública temporal:")
print(public_url)
print("\n📚 Documentación interactiva (Swagger UI):")
print(public_url + "/docs")

## Prueba desde el notebook con `requests`

In [ ]:
import requests
import json

In [ ]:
headers = {"ngrok-skip-browser-warning": "true"}
base_url = public_url

In [ ]:
# 1. Clima
payload = {"message": "¿Qué tiempo hace en Monterrey Nuevo Leon?"}
resp = requests.post(f"{base_url}/agent", json=payload, headers=headers)
print("🔍 Clima:")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

In [ ]:
# 2. Chiste
payload = {"message": "Cuéntame un chiste"}
resp = requests.post(f"{base_url}/agent", json=payload, headers=headers)
print("\n😂 Chiste:")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

In [ ]:
# 3. Pregunta general
payload = {"message": "¿Qué puedes hacer?"}
resp = requests.post(f"{base_url}/agent", json=payload, headers=headers)
print("\n💬 Respuesta general:")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))